In [1]:
import pandas as pd
import numpy as np
import ast

base_path = '../../../../data/preprocessed/'
df = pd.read_csv(f'{base_path}steam_indie_games_graded.csv')
df.columns = df.columns.str.strip()

df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
current_date = pd.to_datetime('2026-05-06')
df['days_since_release'] = (current_date - df['release_date']).dt.days.clip(lower=1)

df['velocity'] = df['total_reviews'] / df['days_since_release']
df['sentiment'] = df['positive_rate'] * np.log1p(df['total_reviews'])
df['value_score'] = df['recommendations_total'].fillna(0) / (df['owners_lower'] + 1)
df['stability'] = df['positive_rate'] / (df['price'] + 1)

mechanics = ['Singleplayer', 'Multiplayer', 'Co-op', 'PvP', 'Online Co-Op', 'Local Co-Op', 'Roguelike', 'Turn-Based', 'Open World', 'Survival', 'Puzzle', 'Platformer', 'Metroidvania', 'Souls-like', 'FPS', 'Tactical', 'Dungeon Crawler', 'Sandbox', 'Crafting', 'Simulation', 'Strategy', 'Action-Adventure', 'Bullet Hell', 'Hack and Slash']
themes = ['Sci-fi', 'Fantasy', 'Horror', 'Historical', 'Cyberpunk', 'Post-apocalyptic', 'Space', 'Medieval', 'Steampunk', 'Zombies', 'Magic', 'War', 'Mystery', 'Lovecraftian', 'Comedy', 'Cute', 'Nature', 'Anime', 'Military']
moods = ['Atmospheric', 'Relaxing', 'Dark', 'Funny', 'Psychological Horror', 'Difficult', 'Casual', 'Emotional', 'Stylized', 'Minimalist', 'Violent', 'Gore', 'Colorful', 'Beautiful', 'Story Rich', 'Surreal']
visuals = ['2D', '3D', 'Pixel Art', 'Pixel Graphics', 'Low-Poly', 'Voxel', 'Hand-drawn', 'Anime', 'Cartoony', 'Realistic', 'Isometric', 'Top-Down', 'Side Scroller', 'First-Person', 'Third Person']

valid_tags = set(mechanics + themes + moods + visuals)

def get_core_tags(tag_str):
    try:
        tags_dict = ast.literal_eval(tag_str)
        return tuple(sorted([t for t in tags_dict.keys() if t in valid_tags]))
    except:
        return tuple()

df['core_tags'] = df['tags'].apply(get_core_tags)
df['tag_count'] = df['core_tags'].apply(len)

def categorize_tag_count(count):
    if count == 1: return '1중'
    elif count == 2: return '2중'
    elif count == 3: return '3중'
    elif count == 4: return '4중'
    elif count == 5: return '5중'
    elif count >= 6: return '5중 이상'
    else: return '0'

df['tag_group'] = df['tag_count'].apply(categorize_tag_count)
df_filtered = df[df['tag_group'] != '0'].copy()

group_stats = df_filtered.groupby('tag_group').agg({
    'velocity': 'median',
    'sentiment': 'median',
    'value_score': 'median',
    'stability': 'median',
    'appid': 'count'
}).rename(columns={'appid': 'game_count'})

order = ['1중', '2중', '3중', '4중', '5중', '5중 이상']
group_stats = group_stats.reindex(order)

print("=== 태그 중첩 개수별 4대 시그널 평균 ===")
print(group_stats)

for i in range(1, 6):
    print(f"\n=== {i}중 태그 Top 5 (velocity 기준, 최소 표본 10개) ===")
    subset = df_filtered[df_filtered['tag_count'] == i]
    combo_stats = subset.groupby('core_tags').agg({
        'velocity': 'median',
        'sentiment': 'median',
        'value_score': 'median',
        'stability': 'median',
        'appid': 'count'
    }).rename(columns={'appid': 'game_count'})
    
    top_combos = combo_stats[combo_stats['game_count'] >= 10].sort_values('velocity', ascending=False).head(5)
    print(top_combos)

=== 태그 중첩 개수별 4대 시그널 평균 ===
           velocity   sentiment  value_score  stability  game_count
tag_group                                                          
1중         0.037594  267.364143          0.0  10.846892          35
2중         0.051761  288.950242          0.0  10.449018          84
3중         0.066932  317.134195          0.0  10.890267         168
4중         0.040889  290.170132          0.0  11.382607         283
5중         0.052224  299.573227          0.0  11.591466         490
5중 이상      0.063601  320.878953          0.0  11.197726        8624

=== 1중 태그 Top 5 (velocity 기준, 최소 표본 10개) ===
                 velocity   sentiment  value_score  stability  game_count
core_tags                                                                
(Singleplayer,)  0.025245  242.227455          0.0  10.361157          12

=== 2중 태그 Top 5 (velocity 기준, 최소 표본 10개) ===
                        velocity   sentiment  value_score  stability  \
core_tags                                 

In [2]:
import pandas as pd
import numpy as np
import ast

base_path = '../../../../data/preprocessed/'
df = pd.read_csv(f'{base_path}steam_indie_games_graded.csv')
df.columns = df.columns.str.strip()

df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
current_date = pd.to_datetime('2026-05-06')
df['days_since_release'] = (current_date - df['release_date']).dt.days.clip(lower=1)

df['velocity'] = df['total_reviews'] / df['days_since_release']
df['sentiment'] = df['positive_rate'] * np.log1p(df['total_reviews'])
df['value_score'] = df['recommendations_total'].fillna(0) / (df['owners_lower'] + 1)
df['stability'] = df['positive_rate'] / (df['price'] + 1)

mechanics = ['Singleplayer', 'Multiplayer', 'Co-op', 'PvP', 'Online Co-Op', 'Local Co-Op', 'Roguelike', 'Turn-Based', 'Open World', 'Survival', 'Puzzle', 'Platformer', 'Metroidvania', 'Souls-like', 'FPS', 'Tactical', 'Dungeon Crawler', 'Sandbox', 'Crafting', 'Simulation', 'Strategy', 'Action-Adventure', 'Bullet Hell', 'Hack and Slash']
themes = ['Sci-fi', 'Fantasy', 'Horror', 'Historical', 'Cyberpunk', 'Post-apocalyptic', 'Space', 'Medieval', 'Steampunk', 'Zombies', 'Magic', 'War', 'Mystery', 'Lovecraftian', 'Comedy', 'Cute', 'Nature', 'Anime', 'Military']
moods = ['Atmospheric', 'Relaxing', 'Dark', 'Funny', 'Psychological Horror', 'Difficult', 'Casual', 'Emotional', 'Stylized', 'Minimalist', 'Violent', 'Gore', 'Colorful', 'Beautiful', 'Story Rich', 'Surreal']
visuals = ['2D', '3D', 'Pixel Art', 'Pixel Graphics', 'Low-Poly', 'Voxel', 'Hand-drawn', 'Anime', 'Cartoony', 'Realistic', 'Isometric', 'Top-Down', 'Side Scroller', 'First-Person', 'Third Person']

def get_4_pillars(tag_str):
    try:
        tags_dict = ast.literal_eval(tag_str)
        tags = list(tags_dict.keys())
        
        m = next((t for t in tags if t in mechanics), None)
        t = next((t for t in tags if t in themes), None)
        mo = next((t for t in tags if t in moods), None)
        v = next((t for t in tags if t in visuals), None)
        
        if m and t and mo and v:
            return f"{m} + {t} + {mo} + {v}"
        return None
    except:
        return None

df['core_4_pillars'] = df['tags'].apply(get_4_pillars)

pillar_analysis = df.dropna(subset=['core_4_pillars']).groupby('core_4_pillars').agg({
    'velocity': 'median',
    'sentiment': 'median',
    'value_score': 'median',
    'stability': 'median',
    'appid': 'count'
}).rename(columns={'appid': 'game_count'})

result = pillar_analysis[pillar_analysis['game_count'] >= 10].sort_values('velocity', ascending=False)
print(result.head(20))

                                        velocity   sentiment  value_score  \
core_4_pillars                                                              
Co-op + Cute + Casual + 3D              0.641962  404.067798     0.003657   
Sandbox + Cute + Funny + 2D             0.473731  556.466986     0.026279   
Puzzle + Horror + Casual + 2D           0.266194  440.841290     0.007855   
Puzzle + Mystery + Casual + 3D          0.201325  376.848758     0.000000   
Bullet Hell + Cute + Casual + 2D        0.199742  373.870434     0.000000   
Co-op + Horror + Dark + 3D              0.177556  324.908066     0.005050   
Puzzle + Magic + Casual + 2D            0.168459  447.666690     0.000000   
Co-op + Comedy + Funny + 3D             0.144608  317.304160     0.000000   
Singleplayer + Cute + Dark + 2D         0.132013  372.602560     0.000000   
Strategy + Cute + Funny + 2D            0.122395  378.622087     0.000000   
Singleplayer + Anime + Casual + 2D      0.114979  343.217199     0.000000   

In [5]:
import pandas as pd
import numpy as np
import ast
from IPython.display import display

base_path = '../../../../data/preprocessed/'
df = pd.read_csv(f'{base_path}steam_indie_games_graded.csv')
df.columns = df.columns.str.strip()

df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
current_date = pd.to_datetime('2026-05-06')
df['days_since_release'] = (current_date - df['release_date']).dt.days.clip(lower=1)

df['velocity'] = df['total_reviews'] / df['days_since_release']
df['sentiment'] = df['positive_rate'] * np.log1p(df['total_reviews'])
df['value_score'] = df['recommendations_total'].fillna(0) / (df['owners_lower'] + 1)
df['stability'] = df['positive_rate'] / (df['price'] + 1)

mechanics = ['Singleplayer', 'Multiplayer', 'Co-op', 'PvP', 'Online Co-Op', 'Local Co-Op', 'Roguelike', 'Turn-Based', 'Open World', 'Survival', 'Puzzle', 'Platformer', 'Metroidvania', 'Souls-like', 'FPS', 'Tactical', 'Dungeon Crawler', 'Sandbox', 'Crafting', 'Simulation', 'Strategy', 'Action-Adventure', 'Bullet Hell', 'Hack and Slash']
themes = ['Sci-fi', 'Fantasy', 'Horror', 'Historical', 'Cyberpunk', 'Post-apocalyptic', 'Space', 'Medieval', 'Steampunk', 'Zombies', 'Magic', 'War', 'Mystery', 'Lovecraftian', 'Comedy', 'Cute', 'Nature', 'Anime', 'Military']
moods = ['Atmospheric', 'Relaxing', 'Dark', 'Funny', 'Psychological Horror', 'Difficult', 'Casual', 'Emotional', 'Stylized', 'Minimalist', 'Violent', 'Gore', 'Colorful', 'Beautiful', 'Story Rich', 'Surreal']
visuals = ['2D', '3D', 'Pixel Art', 'Pixel Graphics', 'Low-Poly', 'Voxel', 'Hand-drawn', 'Anime', 'Cartoony', 'Realistic', 'Isometric', 'Top-Down', 'Side Scroller', 'First-Person', 'Third Person']

def get_4_pillars(tag_str):
    try:
        tags_dict = ast.literal_eval(tag_str)
        tags = list(tags_dict.keys())
        
        m = next((t for t in tags if t in mechanics), None)
        t = next((t for t in tags if t in themes), None)
        mo = next((t for t in tags if t in moods), None)
        v = next((t for t in tags if t in visuals), None)
        
        if m and t and mo and v:
            return f"{m} + {t} + {mo} + {v}"
        return None
    except:
        return None

df['core_4_pillars'] = df['tags'].apply(get_4_pillars)

target_grades = ['high_high', 'high_mid', 'mid_high']
df_target = df[df['performance_grade'].isin(target_grades)].copy()

grade_analysis = df_target.dropna(subset=['core_4_pillars']).groupby(['performance_grade', 'core_4_pillars']).agg({
    'velocity': 'median',
    'sentiment': 'median',
    'value_score': 'median',
    'stability': 'median',
    'appid': 'count'
}).rename(columns={'appid': 'game_count'})

result_frames = []
for grade in target_grades:
    if grade in grade_analysis.index.levels[0]:
        res = grade_analysis.loc[grade].copy()
        res = res[res['game_count'] >= 3].sort_values('velocity', ascending=False).head(10)
        res['performance_grade'] = grade
        result_frames.append(res.reset_index())

final_df = pd.concat(result_frames, ignore_index=True)
final_df = final_df[['performance_grade', 'core_4_pillars', 'velocity', 'sentiment', 'value_score', 'stability', 'game_count']]

display(final_df)

,performance_grade,core_4_pillars,velocity,sentiment,value_score,stability,game_count
0,high_high,Co-op + Cute + Casual + 3D,23.144041,910.461454,0.056850,16.355325,3
1,high_high,Co-op + Horror + Dark + First-Person,13.149675,742.257562,0.040165,8.182019,3
2,high_high,Puzzle + Cute + Story Rich + 2D,7.269874,806.212904,0.096138,6.183721,3
3,high_high,Simulation + Cute + Casual + Anime,5.262943,791.950121,0.036798,9.354905,3
4,high_high,Strategy + Fantasy + Casual + 2D,4.231132,668.336471,0.026610,4.177013,3
5,high_high,Simulation + Anime + Casual + 2D,3.771963,811.604295,0.023870,6.112901,3
6,high_high,Sandbox + Cute + Casual + 3D,3.688213,754.865906,0.031305,12.588927,5
7,high_high,Puzzle + Mystery + Casual + 3D,3.548529,723.706116,0.015520,5.235962,3
8,high_high,Bullet Hell + Cute + Casual + 2D,3.257282,735.466007,0.011535,10.753241,3
9,high_high,Co-op + Horror + Dark + 3D,2.971154,606.749948,0.037648,7.832952,3


In [ ]:
# 사용 안함
import pandas as pd
import numpy as np
import ast
from IPython.display import display

base_path = '../../../../data/preprocessed/'
df = pd.read_csv(f'{base_path}steam_indie_games_graded.csv')
df.columns = df.columns.str.strip()

# 대소문자 표기 오류를 방지하기 위해 전부 소문자로 통일
if 'performance_grade' in df.columns:
    df['performance_grade'] = df['performance_grade'].astype(str).str.lower().str.strip()

df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
current_date = pd.to_datetime('2026-05-06')
df['days_since_release'] = (current_date - df['release_date']).dt.days.clip(lower=1)

df['velocity'] = df['total_reviews'] / df['days_since_release']
df['sentiment'] = df['positive_rate'] * np.log1p(df['total_reviews'])
df['value_score'] = df['recommendations_total'].fillna(0) / (df['owners_lower'] + 1)
df['stability'] = df['positive_rate'] / (df['price'] + 1)

mechanics = ['Singleplayer', 'Multiplayer', 'Co-op', 'PvP', 'Online Co-Op', 'Local Co-Op', 'Roguelike', 'Turn-Based', 'Open World', 'Survival', 'Puzzle', 'Platformer', 'Metroidvania', 'Souls-like', 'FPS', 'Tactical', 'Dungeon Crawler', 'Sandbox', 'Crafting', 'Simulation', 'Strategy', 'Action-Adventure', 'Bullet Hell', 'Hack and Slash']
themes = ['Sci-fi', 'Fantasy', 'Horror', 'Historical', 'Cyberpunk', 'Post-apocalyptic', 'Space', 'Medieval', 'Steampunk', 'Zombies', 'Magic', 'War', 'Mystery', 'Lovecraftian', 'Comedy', 'Cute', 'Nature', 'Anime', 'Military']
moods = ['Atmospheric', 'Relaxing', 'Dark', 'Funny', 'Psychological Horror', 'Difficult', 'Casual', 'Emotional', 'Stylized', 'Minimalist', 'Violent', 'Gore', 'Colorful', 'Beautiful', 'Story Rich', 'Surreal']
visuals = ['2D', '3D', 'Pixel Art', 'Pixel Graphics', 'Low-Poly', 'Voxel', 'Hand-drawn', 'Anime', 'Cartoony', 'Realistic', 'Isometric', 'Top-Down', 'Side Scroller', 'First-Person', 'Third Person']

def get_4_pillars(tag_str):
    try:
        tags_dict = ast.literal_eval(tag_str)
        tags = list(tags_dict.keys())
        
        m = next((t for t in tags if t in mechanics), None)
        t = next((t for t in tags if t in themes), None)
        mo = next((t for t in tags if t in moods), None)
        v = next((t for t in tags if t in visuals), None)
        
        if m and t and mo and v:
            return f"{m} + {t} + {mo} + {v}"
        return None
    except:
        return None

df['core_4_pillars'] = df['tags'].apply(get_4_pillars)

# 전체 등급별 데이터 분포 확인 (터미널 출력용)
print("=== 현재 데이터의 등급별 전체 게임 수 ===")
print(df['performance_grade'].value_counts())
print("-" * 50)

target_grades = ['high_high', 'high_mid', 'mid_high']
df_target = df[df['performance_grade'].isin(target_grades)].copy()

grade_analysis = df_target.dropna(subset=['core_4_pillars']).groupby(['performance_grade', 'core_4_pillars']).agg({
    'velocity': 'median',
    'sentiment': 'median',
    'value_score': 'median',
    'stability': 'median',
    'appid': 'count'
}).rename(columns={'appid': 'game_count'})

result_frames = []
for grade in target_grades:
    if grade in grade_analysis.index.levels[0]:
        res = grade_analysis.loc[grade].copy()
        # 원인 파악을 위해 최소 표본 수를 3개에서 1개로 확 낮춤
        res = res[res['game_count'] >= 1].sort_values('velocity', ascending=False).head(10)
        res['performance_grade'] = grade
        result_frames.append(res.reset_index())

if result_frames:
    final_df = pd.concat(result_frames, ignore_index=True)
    final_df = final_df[['performance_grade', 'core_4_pillars', 'velocity', 'sentiment', 'value_score', 'stability', 'game_count']]
    display(final_df)
else:
    print("조건을 만족하는 데이터가 아예 없어.")

=== 현재 데이터의 등급별 전체 게임 수 ===
performance_grade
low_high     3732
mid_high     2175
low_low       946
high_high     898
low_mid       615
mid_mid       576
mid_low       465
high_mid      182
high_low      103
Name: count, dtype: int64
--------------------------------------------------


,performance_grade,core_4_pillars,velocity,sentiment,value_score,stability,game_count
0,high_high,FPS + Horror + Gore + Realistic,269.938286,1103.448016,0.049738,1.749090,1
1,high_high,Co-op + Horror + Funny + 3D,189.661438,1126.191430,0.013439,10.540200,1
2,high_high,Singleplayer + Cute + Dark + 2D,129.000000,953.492962,0.053294,8.045941,1
3,high_high,FPS + Cute + Casual + 3D,118.691137,913.368513,0.028518,5.517742,2
4,high_high,Co-op + Sci-fi + Casual + 2D,93.321565,1110.580103,0.052789,21.525688,1
5,high_high,Sandbox + Mystery + Casual + 2D,89.106969,1013.313049,0.048013,7.146966,1
6,high_high,Co-op + Horror + Dark + Realistic,84.540404,1037.777882,0.041130,2.278462,1
7,high_high,Sandbox + Magic + Gore + First-Person,83.553779,1059.344666,0.057763,3.119127,1
8,high_high,Co-op + Mystery + Funny + Third Person,77.460641,986.545424,0.025920,15.136824,1
9,high_high,Singleplayer + Cute + Dark + Anime,70.940860,1047.813591,0.048608,12.865202,1


In [8]:
import pandas as pd
import numpy as np
import ast
from scipy import stats

# 1. 데이터 로드 및 전처리
base_path = '../../../../data/preprocessed/'
df = pd.read_csv(f'{base_path}steam_indie_games_graded.csv')
df.columns = df.columns.str.strip()

# 4대 시그널 계산
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
current_date = pd.to_datetime('2026-05-06')
df['days_since_release'] = (current_date - df['release_date']).dt.days.clip(lower=1)

df['velocity'] = df['total_reviews'] / df['days_since_release']
df['sentiment'] = df['positive_rate'] * np.log1p(df['total_reviews'])
df['value_score'] = df['recommendations_total'].fillna(0) / (df['owners_lower'] + 1)
df['stability'] = df['positive_rate'] / (df['price'] + 1)

# 카테고리 정의
mechanics = ['Singleplayer', 'Multiplayer', 'Co-op', 'PvP', 'Online Co-Op', 'Local Co-Op', 'Roguelike', 'Turn-Based', 'Open World', 'Survival', 'Puzzle', 'Platformer', 'Metroidvania', 'Souls-like', 'FPS', 'Tactical', 'Dungeon Crawler', 'Sandbox', 'Crafting', 'Simulation', 'Strategy', 'Action-Adventure', 'Bullet Hell', 'Hack and Slash']
themes = ['Sci-fi', 'Fantasy', 'Horror', 'Historical', 'Cyberpunk', 'Post-apocalyptic', 'Space', 'Medieval', 'Steampunk', 'Zombies', 'Magic', 'War', 'Mystery', 'Lovecraftian', 'Comedy', 'Cute', 'Nature', 'Anime', 'Military']
moods = ['Atmospheric', 'Relaxing', 'Dark', 'Funny', 'Psychological Horror', 'Difficult', 'Casual', 'Emotional', 'Stylized', 'Minimalist', 'Violent', 'Gore', 'Colorful', 'Beautiful', 'Story Rich', 'Surreal']
visuals = ['2D', '3D', 'Pixel Art', 'Pixel Graphics', 'Low-Poly', 'Voxel', 'Hand-drawn', 'Anime', 'Cartoony', 'Realistic', 'Isometric', 'Top-Down', 'Side Scroller', 'First-Person', 'Third Person']

def get_4_pillars(tag_str):
    try:
        tags_dict = ast.literal_eval(tag_str)
        tags = list(tags_dict.keys())
        m = next((t for t in tags if t in mechanics), None)
        t = next((t for t in tags if t in themes), None)
        mo = next((t for t in tags if t in moods), None)
        v = next((t for t in tags if t in visuals), None)
        if m and t and mo and v:
            return f"{m} + {t} + {mo} + {v}"
        return None
    except:
        return None

df['core_4_pillars'] = df['tags'].apply(get_4_pillars)

# 2. 뼈대 조합 분석 (원본 유지)
target_grades = ['high_high', 'high_mid', 'mid_high']
df_target = df[df['performance_grade'].isin(target_grades)].copy()

grade_analysis = df_target.dropna(subset=['core_4_pillars']).groupby(['performance_grade', 'core_4_pillars']).agg({
    'velocity': 'median', 'sentiment': 'median', 'value_score': 'median', 'stability': 'median', 'appid': 'count'
}).rename(columns={'appid': 'game_count'})

result_frames = []
for grade in target_grades:
    if grade in grade_analysis.index.levels[0]:
        res = grade_analysis.loc[grade].copy()
        res = res[res['game_count'] >= 3].sort_values('velocity', ascending=False).head(10)
        res['performance_grade'] = grade
        result_frames.append(res.reset_index())

final_df = pd.concat(result_frames, ignore_index=True)

# 3. 통계 검증 1: 4대 뼈대 조합 자체가 유의미한가? (Kruskal-Wallis)
pillar_counts = df['core_4_pillars'].value_counts()
valid_pillars = pillar_counts[pillar_counts >= 3].index
df_stat = df[df['core_4_pillars'].isin(valid_pillars)]

stat_results = []
for metric in ['velocity', 'sentiment']:
    groups = [df_stat[df_stat['core_4_pillars'] == p][metric].dropna() for p in valid_pillars]
    h_stat, p_val = stats.kruskal(*groups)
    stat_results.append({'Metric': metric, 'p-value': f"{p_val:.4f}", 'Significant': p_val < 0.05})

# 4. 통계 검증 2: 개별 결과의 우연성 검정 (Permutation Test)
perm_results = []
all_vels = df['velocity'].dropna().values
for _, row in final_df.iterrows():
    obs_vel = row['velocity']
    g_count = int(row['game_count'])
    p_chance = np.mean([np.median(np.random.choice(all_vels, g_count)) >= obs_vel for _ in range(1000)])
    perm_results.append(p_chance)

final_df['p_value_chance'] = perm_results
final_df['is_reliable'] = final_df['p_value_chance'] < 0.05

# 결과 출력
print("=== [검증 1] 4대 뼈대 조합의 지표 유의성 ===")
print(pd.DataFrame(stat_results))
print("\n=== [최종 결과] 4대 뼈대 분석 및 신뢰도 검증 ===")
display(final_df[['performance_grade', 'core_4_pillars', 'velocity', 'game_count', 'p_value_chance', 'is_reliable']])

=== [검증 1] 4대 뼈대 조합의 지표 유의성 ===
      Metric p-value  Significant
0   velocity  0.0000         True
1  sentiment  0.0000         True

=== [최종 결과] 4대 뼈대 분석 및 신뢰도 검증 ===


,performance_grade,core_4_pillars,velocity,game_count,p_value_chance,is_reliable
0,high_high,Co-op + Cute + Casual + 3D,23.144041,3,0.000,True
1,high_high,Co-op + Horror + Dark + First-Person,13.149675,3,0.001,True
2,high_high,Puzzle + Cute + Story Rich + 2D,7.269874,3,0.001,True
3,high_high,Simulation + Cute + Casual + Anime,5.262943,3,0.000,True
4,high_high,Strategy + Fantasy + Casual + 2D,4.231132,3,0.001,True
5,high_high,Simulation + Anime + Casual + 2D,3.771963,3,0.006,True
6,high_high,Sandbox + Cute + Casual + 3D,3.688213,5,0.000,True
7,high_high,Puzzle + Mystery + Casual + 3D,3.548529,3,0.005,True
8,high_high,Bullet Hell + Cute + Casual + 2D,3.257282,3,0.006,True
9,high_high,Co-op + Horror + Dark + 3D,2.971154,3,0.001,True
